# Equity Research AI Agent - Examples Notebook

This notebook demonstrates various usage patterns and examples for the Equity Research AI Agent.

## Available Examples:
1. Basic Usage - Simple question/answer
2. Comprehensive Analysis - Using workflows
3. Interactive Q&A - Multiple related questions
4. Specific Analyses - Risk, growth, investment thesis
5. Yahoo Finance Tool - Independent usage
6. Web Search Tool - Independent usage
7. Yahoo Finance RAG Integration - Adding financial data to vector store

## Setup

In [ ]:
import os
from dotenv import load_dotenv

from src.rag.pdf_processor import PDFProcessor
from src.rag.vector_store import VectorStoreManager
from src.agent.equity_research_agent import EquityResearchAgent
from src.agent.workflows import EquityResearchWorkflows
from src.utils.logging_config import setup_logging

# Load environment variables
load_dotenv()
setup_logging(level="INFO")

# Check for API key
if not os.getenv("OPENAI_API_KEY"):
    print("⚠️  Error: OPENAI_API_KEY not found in environment variables.")
    print("Please set it in a .env file or export it as an environment variable.")
else:
    print("✓ Setup complete!")

## Example 1: Basic Usage

Process a PDF and ask a single question.

In [ ]:
print("\n" + "="*80)
print("EXAMPLE 1: Basic Usage")
print("="*80)

# Initialize RAG system
pdf_processor = PDFProcessor()
pdf_data = pdf_processor.process_pdf("Salesforce, Inc. files (10-K) Basic annual filing, for period end 31-Jan-26 (CRM-US).pdf")

vector_store_manager = VectorStoreManager()
documents = vector_store_manager.create_documents_from_pdf_data(pdf_data)
vector_store_manager.create_vector_store_chroma(documents, persist_directory="./chroma_db")

# Initialize agent
agent = EquityResearchAgent(vector_store_manager)

# Ask a question
result = agent.run(
    question="What are Salesforce's main business segments and revenue streams?",
    ticker="CRM"
)

print(f"\nQuestion: {result['question']}")
print(f"\nAnswer:\n{result['answer']}")

## Example 2: Comprehensive Analysis with Workflows

Use predefined workflows for structured analysis.

In [ ]:
print("\n" + "="*80)
print("EXAMPLE 2: Comprehensive Analysis with Workflows")
print("="*80)

# Load existing vector store (assumes Example 1 was run)
vector_store_manager = VectorStoreManager()
vector_store_manager.load_vector_store_chroma("./chroma_db")

# Initialize agent and workflows
agent = EquityResearchAgent(vector_store_manager)
workflows = EquityResearchWorkflows(agent)

# Run comprehensive summary
print("\n--- Comprehensive Summary ---")
summary = workflows.comprehensive_summary(ticker="CRM")
print(summary['answer'][:500] + "...\n")

# Run financial analysis
print("\n--- Financial Analysis ---")
financial = workflows.financial_analysis(ticker="CRM")
print(financial['answer'][:500] + "...\n")

## Example 3: Interactive Q&A Session

Ask multiple related questions in sequence.

In [ ]:
print("\n" + "="*80)
print("EXAMPLE 3: Interactive Q&A Session")
print("="*80)

# Load existing vector store
vector_store_manager = VectorStoreManager()
vector_store_manager.load_vector_store_chroma("./chroma_db")

# Initialize agent and workflows
agent = EquityResearchAgent(vector_store_manager)
workflows = EquityResearchWorkflows(agent)

# Ask multiple related questions
questions = [
    "What were Salesforce's total revenues in the most recent fiscal year?",
    "How does this compare to the previous year?",
    "What are the key drivers of revenue growth?"
]

responses = workflows.q_and_a_session(questions, ticker="CRM")

for i, response in enumerate(responses, 1):
    print(f"\nQuestion {i}: {response['question']}")
    print(f"Answer: {response['answer'][:300]}...\n")
    print("-" * 80)

## Example 4: Specific Analysis Types

Run targeted analyses like risk assessment, growth analysis, and investment thesis.

In [ ]:
print("\n" + "="*80)
print("EXAMPLE 4: Specific Analysis Types")
print("="*80)

# Load existing vector store
vector_store_manager = VectorStoreManager()
vector_store_manager.load_vector_store_chroma("./chroma_db")

# Initialize agent and workflows
agent = EquityResearchAgent(vector_store_manager)
workflows = EquityResearchWorkflows(agent)

# Risk assessment
print("\n--- Risk Assessment ---")
risks = workflows.risk_assessment(ticker="CRM")
print(risks['answer'][:500] + "...\n")

# Growth analysis
print("\n--- Growth Analysis ---")
growth = workflows.growth_analysis(ticker="CRM")
print(growth['answer'][:500] + "...\n")

# Investment thesis
print("\n--- Investment Thesis ---")
thesis = workflows.investment_thesis(ticker="CRM")
print(thesis['answer'][:500] + "...\n")

## Example 5: Yahoo Finance Tool (Independent Usage)

Use the Yahoo Finance tool without the agent or vector store.

In [ ]:
print("\n" + "="*80)
print("EXAMPLE 5: Yahoo Finance Tool (Independent Usage)")
print("="*80)

from src.tools.yahoo_finance import YahooFinanceTool

yf_tool = YahooFinanceTool()

# Get stock info
print("\n--- Stock Information ---")
info = yf_tool.get_stock_info("CRM")
print(f"Company: {info['name']}")
print(f"Sector: {info['sector']}")
print(f"Current Price: ${info['current_price']}")
print(f"Market Cap: ${info['market_cap']:,}" if isinstance(info['market_cap'], (int, float)) else f"Market Cap: {info['market_cap']}")
print(f"P/E Ratio: {info['pe_ratio']}")

# Get key metrics summary
print("\n--- Key Metrics Summary ---")
summary = yf_tool.get_key_metrics_summary("CRM")
print(summary[:500] + "...")

## Example 6: Web Search Tool (Independent Usage)

Use the web search tool to find recent news and information.

In [ ]:
print("\n" + "="*80)
print("EXAMPLE 6: Web Search Tool (Independent Usage)")
print("="*80)

from src.tools.web_search import WebSearchTool

search_tool = WebSearchTool(max_results=3)

# Search for company news
print("\n--- Company News Search ---")
news = search_tool.search_company_news("Salesforce", "CRM")
print(news[:500] + "...")

## Example 7: Yahoo Finance RAG Integration

Demonstrate how Yahoo Finance data is converted to documents and can be added to the RAG system.

In [ ]:
print("\n" + "="*80)
print("EXAMPLE 7: Yahoo Finance RAG Integration")
print("="*80)

from src.tools.yahoo_finance import YahooFinanceTool
from src.rag.vector_store import VectorStoreManager

# Initialize Yahoo Finance tool
yf_tool = YahooFinanceTool()

# Create documents from Yahoo Finance data
print("\n--- Creating Documents from Yahoo Finance Data ---")
ticker = "CRM"
finance_docs = yf_tool.create_documents_from_financial_data(ticker)

print(f"Created {len(finance_docs)} documents from Yahoo Finance data for {ticker}")
print("\nDocument types:")
for doc in finance_docs:
    content_type = doc.metadata.get('content_type', 'unknown')
    source = doc.metadata.get('source', 'unknown')
    print(f"  - {content_type} (Source: {source})")
    print(f"    Preview: {doc.page_content[:100]}...")

# Demonstrate adding to vector store
print("\n--- Adding to Vector Store ---")
print("These documents can be added to an existing vector store using:")
print("  vector_store_manager.add_documents(finance_docs)")
print("\nOnce added, the financial data becomes searchable via RAG alongside PDF documents.")
print("The agent can retrieve specific metrics like P/E ratio, market cap, etc., through semantic search.")

## Additional Examples

You can combine the above patterns to create custom workflows:

In [ ]:
# Example: Full workflow with all features
print("\n" + "="*80)
print("CUSTOM WORKFLOW: Complete Analysis")
print("="*80)

# 1. Load vector store
vector_store_manager = VectorStoreManager()
vector_store_manager.load_vector_store_chroma("./chroma_db")

# 2. Initialize agent
agent = EquityResearchAgent(vector_store_manager)
workflows = EquityResearchWorkflows(agent)

# 3. Run multiple analyses
analyses = [
    ("Comprehensive Summary", workflows.comprehensive_summary),
    ("Financial Analysis", workflows.financial_analysis),
    ("Risk Assessment", workflows.risk_assessment),
    ("Growth Analysis", workflows.growth_analysis),
]

for name, analysis_func in analyses:
    print(f"\n--- {name} ---")
    result = analysis_func(ticker="CRM")
    print(result['answer'][:300] + "...\n")
    print("-" * 80)

## Tips for Using These Examples

1. **Run examples sequentially**: Example 1 creates the vector store needed by other examples
2. **Modify parameters**: Change ticker symbols, questions, and analysis types as needed
3. **Combine patterns**: Mix and match different approaches for custom workflows
4. **Use existing data**: Set `use_existing=True` to skip PDF reprocessing
5. **Check API quotas**: Yahoo Finance and OpenAI have rate limits

## Next Steps

- Modify the questions and analyses to suit your needs
- Add your own PDF documents for analysis
- Explore different ticker symbols
- Create custom workflows using the agent API